In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
from synphot import SpectralElement
from synphot.models import Empirical1D
import os, glob
import emcee
import progressbar
import sys
import astropy.units as u
import astropy.constants as const
import traceback
import pickle
from astropy.io import fits, ascii
from astropy.stats import sigma_clipped_stats as scs
from scipy import interpolate
from scipy.integrate import simpson
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.io.misc.hdf5 import read_table_hdf5
import pandas as pd
import itertools
from tqdm import tqdm
import corner
import time
from scipy.stats import gaussian_kde
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error as rmse
import torch
from sbi import utils as sbi_utils
from sbi.neural_nets import posterior_nn
from sbi import inference
from torch.distributions import MultivariateNormal, Exponential, LogNormal
from sbi.utils import MultipleIndependent, BoxUniform
import signal

import rsg_phot.dust as dust
from rsg_phot.mcmc import mcmc
from rsg_phot.mc_parallel import rsg_dataloader, mcmcfit
from rsg_phot.rsg_sbi import sbifit
from rsg_phot import sbi_pp

In [ ]:
nrc_filts = np.array(['F070W','F090W','F115W','F140M','F150W', 'F150W2', 'F162M',
                        'F164N','F182M','F187N','F200W','F210M','F212N','F250M',
                        'F277W','F300M','F322W2','F323N','F335M','F356W','F360M',
                        'F405N','F410M','F430M','F444W','F460M','F466N','F470N','F480M'])
miri_filts = np.array(['F0560W','F0770W','F1000W','F1130W','F1280W','F1500W',
                           'F1800W','F2100W','F2550W'])
jwst_filts = np.concatenate((nrc_filts, miri_filts))
wv_all = [float(i.replace('F', '').replace('W2', '').replace('M', '').replace('N', '').replace('W', ''))/100 
          for i in jwst_filts]

### RSG grid

In [ ]:
comp = 'sil'
bandpass_dir = os.environ['BANDPASS']

In [ ]:
temps = np.array([2600.0, 2800.0, 3000.0, 3200.0, 3400.0, 3600.0, 3800.0, 4000.0, 4200.0, 4500.0, 4800.0, 5000.0])
dust_temps = np.array([500.0, 800.0, 1000.0, 1200.0, 1500.0])
taus = np.array([0.01, 0.325, 0.641, 0.956, 1.27, 1.59, 1.9, 2.22, 2.53, 2.85, 3.16, 3.48, 3.79,
                 4.11, 4.42, 4.74, 5.05, 5.37, 5.68, 6.0])
lums = np.logspace(3, 6, 16)
rv = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
av = np.arange(0.0, 5.5, 0.5)
len(temps), len(dust_temps), len(taus), len(lums), len(rv), len(av)

In [ ]:
full_grid = np.zeros((1497, len(temps), len(dust_temps), len(taus), len(lums), len(rv), len(av)))
mags = {}
for flt in nrc_filts:
    mags[flt] = np.zeros((len(temps), len(dust_temps), len(taus), len(lums), len(rv), len(av)))
len(full_grid.flatten())/1e6

In [ ]:
spec = read_table_hdf5('data/r2/dusty_sil_grid/dusty_out/rsg_2600.0_500.0/rsg_2600.0_500.0.hdf5')
# spec = read_table_hdf5('data/r2/dusty_sil_grid/dusty_out/rsg_3000.0_800.0/rsg_3000.0_800.0.hdf5')

In [ ]:
plt.plot(spec['lambda'], spec['flam_0.01'])
# plt.axvline(10)
plt.xscale('log')
plt.yscale('log')
# plt.ylim(1e-4, 1e-1)
# plt.xlim(5, 15)

In [ ]:
with open(os.path.join('data', 'interpolate', 'rsg_s2.pkl'), 'rb') as f:
    models = pickle.load(f)

with open(os.path.join('data', 'interpolate', 'rsg_s2_test.pkl'), 'rb') as f:
    models_test = pickle.load(f)

In [ ]:
tg, dtg, tug, lg = np.meshgrid(2800, 800, .01, 1e4, indexing='ij', sparse=True)
models['F070W']((tg, dtg, tug, lg)).flatten(), models_test['F070W']((tg, dtg, tug, lg)).flatten()

In [ ]:
interp_val, interp_test = [], []
nrc_s = ['F070W', 'F090W', 'F115W', 'F150W', 'F200W', 'F250M', 'F277W', 'F356W', 'F444W']
wv_s = [float(i.split('F')[1].split('W')[0].split('M')[0]) for i in nrc_s]

wv_all = [float(i.replace('F', '').replace('W2', '').replace('M', '').replace('N', '').replace('W', ''))/100 
          for i in nrc_filts]
for f_ in nrc_filts:
    interp_val.append(models[f_]((tg, dtg, tug, lg)).flatten())
    interp_test.append(models_test[f_]((tg, dtg, tug, lg)).flatten())

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, sharex=True, gridspec_kw={'height_ratios': [3, 1]})
ax1.scatter(wv_all, interp_val, fc='orange', ec='black')
ax1.scatter(wv_all, interp_test, fc='red', ec='black')
ax1.invert_yaxis()

ax2.scatter(wv_all, np.array(interp_val)-np.array(interp_test), fc='orange', ec='black')
ax2.axhline(0.0, ls='--', color='black', alpha=0.3)
ax2.set_ylim(-0.1, 0.1)

fig.subplots_adjust(hspace=0.1)

In [ ]:
with open('/Users/aswin/rsg_phot/data/interpolate/rsg_sil_r2_z0.00_ext.pkl', 'rb') as f:
    z0 = pickle.load(f)

with open('/Users/aswin/rsg_phot/data/interpolate/rsg_sil_r2_z-0.25_ext.pkl', 'rb') as f:
    z25 = pickle.load(f)

with open('/Users/aswin/rsg_phot/data/interpolate/rsg_sil_r2_z-0.50_ext.pkl', 'rb') as f:
    z50 = pickle.load(f)

theta = (4000, 800, 0.01, 1e4, 3.1, 0.5)
params = np.meshgrid(*theta, indexing='ij', sparse=True)
mz0 = np.array([z0[f](params).flatten()[0] for f in nrc_filts])
mz25 = np.array([z25[f](params).flatten()[0] for f in nrc_filts])
mz50 = np.array([z50[f](params).flatten()[0]+10 for f in nrc_filts])

plt.scatter(wv_all, mz0, s=50, ec='black', label='logZ=0.0')
plt.scatter(wv_all, mz25, s=50, ec='black', label='logZ=-0.25')
plt.scatter(wv_all, mz50, s=50, ec='black', label='logZ=-0.50')
plt.legend()
plt.xlabel('Wavelength (um)')
plt.gca().invert_yaxis()

In [ ]:
dustgen = dust.dusty_gen(modeldir='/Users/aswin/rsg_phot/data/marcs/m1.00_g0.00/')
dustgen.gen_grid(grid_temps=[4000.0], grid_dust_temps=[800.0], redo_dusty=True, tb_overwrite=True)

In [ ]:
model = '../data/interpolate/miri/MARCS_Z+0.00_sil.pkl'
with open(model, 'rb') as f:
    miri_model = pickle.load(f)

plt.figure()
theta = (4000, 800, 0.01, 4, 3.1, 0.5)
params = np.meshgrid(*theta, indexing='ij', sparse=True)
m0 = np.array([miri_model[f](params).flatten()[0] for f in jwst_filts])
plt.plot(wv_all, m0, ls='--', marker='o', mec='black', label='tau=0.01')

theta = (4000, 800, 0.1, 4, 3.1, 0.5)
params = np.meshgrid(*theta, indexing='ij', sparse=True)
m1 = np.array([miri_model[f](params).flatten()[0] for f in jwst_filts])
plt.plot(wv_all, m1, ls='--', marker='o', mec='black', label='tau=0.1')

theta = (4000, 800, 1.0, 4, 3.1, 0.5)
params = np.meshgrid(*theta, indexing='ij', sparse=True)
m2 = np.array([miri_model[f](params).flatten()[0] for f in jwst_filts])
plt.plot(wv_all, m2, ls='--', marker='o', mec='black', label='tau=1.0')

theta = (4000, 800, 12.0, 4, 3.1, 0.5)
params = np.meshgrid(*theta, indexing='ij', sparse=True)
m3 = np.array([miri_model[f](params).flatten()[0] for f in jwst_filts])
plt.plot(wv_all, m3, ls='--', marker='o', mec='black', label='tau=2.0')

plt.legend()
plt.xlabel('Wavelength (um)')
plt.gca().invert_yaxis()

### mcmc

In [ ]:
df_sil = pd.read_csv('data/dolphot/ngc5643/ngc5643_sil_rsgcat.csv')
df_grf = pd.read_csv('data/dolphot/ngc5643/ngc5643_grf_rsgcat.csv')
df_sil.set_index('index', inplace=True)
df_grf.set_index('index', inplace=True)

grfm = np.array([i in df_sil.index for i in df_grf.index])
silm = np.array([i in df_grf.index for i in df_sil.index])

df1 = df_sil[~silm]
df2 = df_grf[~grfm]
df3 = df_sil[silm][df_sil[silm]['chimin'].values < df_grf[grfm]['chimin'].values]
df4 = df_grf[grfm][df_sil[silm]['chimin'].values > df_grf[grfm]['chimin'].values]
rsgcat = pd.concat([df1, df2, df3, df4])

In [ ]:
load_args = {
    'gal':'ngc5643', 'procdir':'data/dolphot/ngc5643', 'photfile_path':'data/dolphot/ngc5643',
    'dm':30.57, 'dmerr':0.06, 'z':-0.5, 'trgb':('F090W', 26.20),
    'modeltype':'MARCS', 'comp':'sil',
    'keep_narrow':False, 'agbcut':False,
    'rsgcat':rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
rsgloader.plot_error_dist()

In [ ]:
sedfit = mcmcfit(rsgloader, ncores=1, keep_narrow=False, modeltype='MARCS', comp='sil', verbose=True)
col = rsgcat.sample(1).iloc[0]
mc_fit_params, min_chi_params, chi_posterior, chi_best = sedfit.run_mcmc(col, nsteps=350, nwalkers=64, burn_in=75, plot=True)

In [ ]:
print('chisq params:', col[['teff_chisq', 'lum_chisq', 'tdust_chisq', 'tau_chisq', 'av_chisq', 'chimin']].values)
print('posterior params:', np.array(list(mc_fit_params.values()))[:, 0])
print('chi min params:', min_chi_params)

In [ ]:
labels = sedfit.mc_obj.model_fit_params
reader = sedfit.mc_obj.load_backend(rsgloader.gen_phot(col))
sample = np.array(reader.get_chain(flat=True, discard=75, thin=1))

fig, axes = plt.subplots(len(labels), figsize=(10, 7), sharex=True)
for i in range(len(labels)):
    ax = axes[i]
    ax.plot(sample[:, i], "k", alpha=0.3)
    ax.set_xlim(0, len(sample))
    ax.set_ylabel(labels[i])
    # ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number");

### SBI

In [ ]:
rsgcat = pd.read_csv(os.path.join(os.pardir, 'data/dolphot/ngc4449/ngc4449_ngc4485_sil_rsgcat.csv'))
load_args = {
    'gal':'ngc4449', 'procdir':os.path.join(os.pardir, 'data/dolphot/ngc4449'), 'photfile_path':None,
    'dm':28.02, 'dmerr':0.32, 'z':-0.50, 'trgb':('F090W', 25.11),
    'modeltype':'MARCS', 'comp':'sil', 'keep_narrow':False,
    'rsgcat':rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
rsgcat = pd.read_csv(os.path.join(os.pardir, 'data/dolphot/ngc5643/ngc5643_sil_rsgcat.csv'))
load_args = {
    'gal':'ngc5643', 'procdir':os.path.join(os.pardir, 'data/dolphot/ngc5643'), 'photfile_path':'data/dolphot/ngc5643',
    'dm':30.57, 'dmerr':0.00, 'z':-0.25, 'trgb':('F090W', 26.20),
    'modeltype':'MARCS', 'comp':'sil',
    'keep_narrow':False, 'agbcut':False, 'ignore_filts':['F300M'],
    'rsgcat':rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
rsgcat = pd.read_csv(os.path.join(os.pardir, 'data/dolphot/ngc4485/ngc4485_sil_rsgcat.csv'))
load_args = {
    'gal':'ngc4485', 'procdir':os.path.join(os.pardir, 'data/dolphot/ngc4485'), 'photfile_path':None,
    'dm':29.67, 'dmerr':0.5, 'z':-0.5, 'trgb':('F090W', 26.62),
    'modeltype':'MARCS', 'comp':'sil',
    'keep_narrow':False, 'agbcut':False, 'ignore_filts':None,
    'rsgcat':rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
rsgcat = pd.read_csv(os.path.join(os.pardir, 'data/dolphot/ngc4258/ngc4258_sil_rsgcat.csv'))
load_args = {
    'gal':'ngc4258', 'procdir':os.path.join(os.pardir, 'data/dolphot/ngc4258'), 'photfile_path':None,
    'dm':29.397, 'dmerr':0.03, 'z':-0.25, 'trgb':('F090W', 25.055),
    'modeltype':'MARCS', 'comp':'sil',
    'keep_narrow':False, 'agbcut':False,
    'rsgcat':rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
rsgcat = pd.read_csv('../data/dolphot/ngc1365/ngc1365_sil_rsgcat.csv')
load_args = {
    'gal':'ngc1365', 'procdir':'../data/dolphot/ngc1365', 
    'photfile_path':None,
    'dm':31.29, 'dmerr':0.065, 'z':-0.25, 'trgb':('F090W', 27.34),
    'modeltype':'MARCS', 'comp':'sil',
    'keep_narrow':False,
    'rsgcat':rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
rsgcat = pd.read_csv('../data/dolphot/ngc4536/ngc4536_sil_rsgcat.csv')
load_args = {
    'gal':'ngc4536', 'procdir':'../data/dolphot/ngc4536', 
    'dm':30.99, 'dmerr':0.06, 'z':-0.25, 'trgb':('F090W', 27.01),
    'modeltype':'MARCS', 'comp':'sil', 'ignore_filts': ['F115W', 'F444W'],
    'keep_narrow':False, 'rsgcat': rsgcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
sedfit = sbifit(rsgloader)
sedfit_mc = mcmcfit(rsgloader, ncores=1, verbose=True)

m, f, t, n = 'nsf', 10, 5, 10
savepath = f'../data/dolphot/{rsgloader.gal}/sbi/npe_{m}_{f}_{t}_{n}.pt'
hatp_x_y = sedfit.baseline_sbi_model(flow_model=m, hidden_features=f, ntransforms=t, nbins=n)
                                    #  augment_train=True, augment_size=500000, batch_size=512)

sedfit.model_info(sedfit.savepath)

In [ ]:
sedfit.run_calibration(num_sbc_samples=500, num_posterior_samples=2500, num_test_samples=1000)

In [ ]:
#SBC
from sbi.analysis.plot import sbc_rank_plot
from sbi.diagnostics import check_sbc, check_tarp, run_sbc, run_tarp

num_sbc_samples = 500

def simulator(theta_in):
    theta_in[:, 0] = theta_in[:, 0]*1e3
    theta_in[:, 1] = theta_in[:, 1]*1e3
    out = np.zeros((len(theta_in), len(sedfit.rsgloader.cols['flts'][sedfit.rsgloader.flt_mask])))

    for i, theta in enumerate(theta_in):
        model_mag = np.array([sedfit.gen_mc_obj.model[f](theta).flatten()[0] for f in sedfit.rsgloader.cols['flts'][sedfit.rsgloader.flt_mask]]) + sedfit.gen_mc_obj.dm
        out[i, :] = model_mag

    outdf = pd.DataFrame(out, columns=sedfit.rsgloader.cols['flts'][sedfit.rsgloader.flt_mask])
    noise = sedfit.sim_skew_mag_err(outdf) 

    out = np.hstack((out, noise))
    out = torch.as_tensor(out.astype(np.float32)).to('cpu')
    return out

# generate ground truth parameters and corresponding simulated observations for SBC.
thetas = sedfit.prior.sample((num_sbc_samples,))
# thetas = thetas[thetas[:, 3] < 4.2]
print(f'Running SBC on {len(thetas)} samples')
xs = simulator(thetas.clone().detach())

num_posterior_samples = 2500
num_workers = 1
ranks, dap_samples = run_sbc(
    thetas,
    xs,
    hatp_x_y,
    num_posterior_samples=num_posterior_samples,
    num_workers=num_workers,
    use_sample_batched=False,  # True can give a speed-up, but might cause memory issues.
)

check_stats = check_sbc(
    ranks, thetas, dap_samples, num_posterior_samples=num_posterior_samples
)

print(check_stats)

f, ax = sbc_rank_plot(
    ranks=ranks,
    num_posterior_samples=num_posterior_samples,
    plot_type="hist",
    num_bins=None,  # by passing None we use a heuristic for the number of bins.
    parameter_labels=sedfit.gen_mc_obj.model_fit_params,
)

f, ax = sbc_rank_plot(ranks, num_posterior_samples, plot_type="cdf", parameter_labels=sedfit.gen_mc_obj.model_fit_params)

In [ ]:
#TARP
ecp, alpha = run_tarp(
    thetas,
    xs,
    hatp_x_y,
    references=None,  # will be calculated automatically.
    num_posterior_samples=2500,
)

atc, ks_pval = check_tarp(ecp, alpha)
print(atc, "Should be close to 0")
print(ks_pval, "Should be larger than 0.05")

from sbi.analysis.plot import plot_tarp

plot_tarp(ecp, alpha);

In [ ]:
mcol_, ecol_ = rsgloader.cols['magcols'][rsgloader.flt_mask], rsgloader.cols['errcols'][rsgloader.flt_mask]
detm = ((rsgloader.rsgcat[mcol_] > 10) & (rsgloader.rsgcat[mcol_] < 35)).all(axis=1).values
#4800
col = rsgloader.rsgcat[detm].sample(1)
print('Index:', col.index[0])
obsmag = col[mcol_].iloc[0].to_numpy() - (rsgloader.gen_mc_obj.dm+30)
obserr = col[ecol_].iloc[0].to_numpy()
obserr = np.sqrt(obserr**2 + 0.01**2)
obs = np.concatenate([obsmag, obserr])
hatp_x_y.set_default_x(torch.as_tensor(obs.astype(np.float32)).to('cpu'))
samp = hatp_x_y.sample((2500,), show_progress_bars=True)
lp = hatp_x_y.log_prob(samp)

In [ ]:
for i in xs.numpy():
    plt.plot(i[:9], color='gray', alpha=0.2)

plt.plot(obs[:9], color='crimson')
plt.gca().invert_yaxis()

plt.figure()
for i in xs.numpy():
    plt.plot(i[9:], color='gray', alpha=0.2)

plt.plot(obs[9:], color='crimson')
plt.yscale('log')

In [ ]:
# corner plot of trained model on posterior samples drawn on training data
yt = sedfit.y_train
xt = sedfit.x_train
idx = np.random.choice(len(yt))
i = torch.as_tensor(yt[idx].astype(np.float32)).to('cpu')
i = i + torch.rand_like(i)*1e-2
# i[9:] = i[9:]*4
hatp_x_y.set_default_x(i)
samp = hatp_x_y.sample((25000,), show_progress_bars=True)
lp = hatp_x_y.log_prob(samp)
samp = samp[lp > torch.quantile(lp, 0.1)]
corner.corner(samp.numpy(), truths=xt[idx], show_titles=True, title_fmt='.2f', title_kwargs={"fontsize": 12},
              quantiles=[0.16, 0.5, 0.84]);

In [ ]:
# accuracy check on test data

truths = []
est = []
err16, err84 = [], []
for _ in tqdm(range(1000)):
    idx = np.random.choice(len(xs))
    i = xs[idx]
    # i = i + torch.rand_like(i)*1e-2
    hatp_x_y.set_default_x(i)
    samp = hatp_x_y.sample((2500,), show_progress_bars=False)
    lp = hatp_x_y.log_prob(samp)
    samp = samp[lp > torch.quantile(lp, 0.1)]
    samp = samp.numpy()
    med = np.median(samp, axis=0)
    p16, p84 = np.percentile(samp, 16, axis=0), np.percentile(samp, 84, axis=0)
    truths.append(thetas[idx])
    est.append(med)
    err16.append(p16); err84.append(p84)

truths = np.array(truths)
est = np.array(est)
err16 = np.array(err16)
err84 = np.array(err84)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
labels = sedfit.gen_mc_obj.model_fit_params
for i in range(6):  
    ax = axes.flatten()[i]
    ax.errorbar(truths[:, i], est[:, i], yerr=[est[:, i]-err16[:, i], err84[:, i]-est[:, i]], fmt='o', 
                ecolor='gray', alpha=0.5, label='Estimated', markersize=4)
    ax.scatter(truths[:, i], truths[:, i], color='red', label='Truth', s=10)
    ax.plot([truths[:, i].min(), truths[:, i].max()], [truths[:, i].min(), truths[:, i].max()], 
            ls='--', color='black', alpha=0.7)
    r2, rms = r2_score(truths[:, i], est[:, i]), rmse(truths[:, i], est[:, i])
    ax.annotate(f'R2: {r2:.3f}\nRMSE: {rms:.3f}', xy=(0.7, 0.05), xycoords='axes fraction')
    ax.set_xlabel('Simulated ' + labels[i])
    ax.set_ylabel('Estimated ' + labels[i])
    ax.legend(loc='upper left')

    f = (truths[:, i] < err84[:, i]) & (truths[:, i] > err16[:, i])
    print(f'Parameter: {labels[i]}, Coverage: {np.sum(f)}/{len(f)} = {np.sum(f)/len(f):.3f}')

In [ ]:
# accuracy check on training data

truths = []
est = []
err16, err84 = [], []
for _ in tqdm(range(2000)):
    idx = np.random.choice(len(yt))
    i = torch.as_tensor(yt[idx].astype(np.float32)).to('cpu')
    # i = i + torch.rand_like(i)*1e-2
    hatp_x_y.set_default_x(i)
    samp = hatp_x_y.sample((2500,), show_progress_bars=False)
    lp = hatp_x_y.log_prob(samp)
    samp = samp[lp > torch.quantile(lp, 0.1)]
    samp = samp.numpy()
    med = np.median(samp, axis=0)
    p16, p84 = np.percentile(samp, 16, axis=0), np.percentile(samp, 84, axis=0)
    truths.append(xt[idx])
    est.append(med)
    err16.append(p16); err84.append(p84)

truths = np.array(truths)
est = np.array(est)
err16 = np.array(err16)
err84 = np.array(err84)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
labels = sedfit.gen_mc_obj.model_fit_params
for i in range(6):  
    ax = axes.flatten()[i]
    ax.errorbar(truths[:, i], est[:, i], yerr=[est[:, i]-err16[:, i], err84[:, i]-est[:, i]], fmt='o', 
                ecolor='gray', alpha=0.5, label='Estimated', markersize=4)
    ax.scatter(truths[:, i], truths[:, i], color='red', label='Truth', s=10)
    ax.plot([truths[:, i].min(), truths[:, i].max()], [truths[:, i].min(), truths[:, i].max()], 
            ls='--', color='black', alpha=0.7)
    r2, rms = r2_score(truths[:, i], est[:, i]), rmse(truths[:, i], est[:, i])
    ax.annotate(f'R2: {r2:.3f}\nRMSE: {rms:.3f}', xy=(0.7, 0.05), xycoords='axes fraction')
    ax.set_xlabel('Simulated ' + labels[i])
    ax.set_ylabel('Estimated ' + labels[i])
    ax.legend(loc='upper left')

    f = (truths[:, i] < err84[:, i]) & (truths[:, i] > err16[:, i])
    print(f'Parameter: {labels[i]}, Coverage: {np.sum(f)}/{len(f)} = {np.sum(f)/len(f):.3f}')

In [ ]:
# compare sbi with mcmc
def sbi_samples_corner(samp, lp, sedfit, fig):
    samples = samp.cpu().numpy() if hasattr(samp, "cpu") else np.asarray(samp)
    lp = lp.cpu().numpy() if hasattr(lp, "cpu") else np.asarray(lp)
    lp = 10**lp
    medians = np.percentile(samp, 50, weights=lp, method='inverted_cdf', axis=0)
    p16 = np.percentile(samp, 16, weights=lp, method='inverted_cdf', axis=0)
    p84 = np.percentile(samp, 84, weights=lp, method='inverted_cdf', axis=0)
    ndim = samples.shape[1]
    labels = sedfit.gen_mc_obj.model_fit_params

    # make corner plot
    fig_sbi = corner.corner(samples, fig=fig, labels=labels, color='C1', hist_kwargs={'density': True}, 
                            label_kwargs={'fontsize': 10}, show_titles=False, title_fmt='.3f', plot_density=False,
                            plot_datapoints=False, fill_contours=False, truth_color='crimson', truths=medians)
    # arrange axes and annotate means and 1-sigma
    axes = np.array(fig_sbi.axes).reshape((ndim, ndim))
    for i in range(ndim):
        m_i, p16_i, p84_i = medians[i], p16[i], p84[i]
        # diagonal: add vertical lines and title with mean +/- sigma
        ax_diag = axes[i, i]
        ax_diag.axvline(p16[i], color='crimson', lw=1, ls='--')
        ax_diag.axvline(p84[i], color='crimson', lw=1, ls='--')
        ax_diag.set_title(r"${0:.3f}^{{+{1:.3f}}}_{{-{2:.3f}}}$".format(m_i, p84_i-m_i, m_i-p16_i), fontsize=9)

        # off-diagonals: mark the mean point
        for j in range(i):
            ax = axes[i, j]
            ax.scatter(medians[j], medians[i], marker='x', color='k', s=30)

    fig_sbi.suptitle("Posterior comparison", fontsize=12)
    plt.show()

mcol_, ecol_ = rsgloader.cols['magcols'][rsgloader.flt_mask], rsgloader.cols['errcols'][rsgloader.flt_mask]
detm = ((rsgloader.rsgcat[mcol_] > 10) & (rsgloader.rsgcat[mcol_] < 35)).all(axis=1).values
col = rsgloader.rsgcat[detm].sample(1)
print('Index:', col.index[0])

#run mcmc
testcol = rsgcat.loc[col.index[0]]
mc_fit_params, min_chi_params, chi_posterior, chi_best = sedfit_mc.run_mcmc(testcol, nsteps=350, nwalkers=64, burn_in=75, plot=False)
phot_mc = rsgloader.gen_phot(testcol)
fig = sedfit_mc.plot_corner(phot_mc, min_chi_params)

# sbi obs
temper = 1.0
obsmag = col[mcol_].iloc[0].to_numpy() - (rsgloader.gen_mc_obj.dm+30)
obserr = col[ecol_].iloc[0].to_numpy() * 2
obserr = np.sqrt(obserr**2 + 0.01**2)
obs = np.concatenate([obsmag, obserr])
hatp_x_y.set_default_x(torch.as_tensor(obs.astype(np.float32)).to('cpu'))
if temper != 1.0:
    samp = hatp_x_y.sample((20000,), show_progress_bars=True)
    lp = hatp_x_y.log_prob(samp) 
    lp_temp = lp / temper
    weights = torch.exp(lp_temp - lp)
    indices = torch.multinomial(weights, num_samples=10000, replacement=True)
    samp = samp[indices]
    lp = lp[indices]
else:
    samp = hatp_x_y.sample((2500,), show_progress_bars=True)
    lp = hatp_x_y.log_prob(samp)
    samp = samp[lp > torch.quantile(lp, 0.1)]
    lp = lp[lp > torch.quantile(lp, 0.1)]

# samp[:, 0] = samp[:, 0]*1e3
# samp[:, 1] = samp[:, 1]*1e3

sbi_samples_corner(samp, lp, sedfit, fig)

In [ ]:
#posterior predictive check
mc_fitpe = np.array(list(mc_fit_params.values()))
mc_params = np.meshgrid(*mc_fitpe[:, 0], indexing='ij', sparse=True)
model_mag_mc = np.array([sedfit.gen_mc_obj.model[f](mc_params).flatten()[0] for f in sedfit.rsgloader.cols['flts'][sedfit.rsgloader.flt_mask]]) + sedfit.gen_mc_obj.dm
mmp = np.array([sedfit.gen_mc_obj.model[f](mc_params).flatten()[0] for f in sedfit.rsgloader.nrc_filts]) + sedfit.gen_mc_obj.dm
chi_mc = ((phot_mc['mag'] - rsgloader.dm) - model_mag_mc)**2/(phot_mc['magerr'])**2
chi_mc = np.sum(chi_mc)

sbi_fitpe = np.median(samp.numpy(), axis=0)
sbi_params = np.meshgrid(*sbi_fitpe, indexing='ij', sparse=True)
model_mag_sbi = np.array([sedfit.gen_mc_obj.model[f](sbi_params).flatten()[0] for f in sedfit.rsgloader.cols['flts'][sedfit.rsgloader.flt_mask]]) + sedfit.gen_mc_obj.dm
smp = np.array([sedfit.gen_mc_obj.model[f](sbi_params).flatten()[0] for f in sedfit.rsgloader.nrc_filts]) + sedfit.gen_mc_obj.dm
chi_sbi = ((phot_mc['mag'] - rsgloader.dm) - model_mag_sbi)**2/(phot_mc['magerr'])**2
chi_sbi = np.sum(chi_sbi)

fig, (ax1, ax2) = plt.subplots(nrows=2, sharex=True, gridspec_kw={'height_ratios': [3, 1]})
plt.subplots_adjust(hspace=0.1)
wv = np.array([float(i[1:4])/100 for i in phot_mc['inst_filt']])
wv2 = sedfit.rsgloader.wv_all
ax1.errorbar(x=wv, y=phot_mc['mag']-rsgloader.dm, yerr=phot_mc['magerr'], linestyle='none', marker='o', markerfacecolor='cornflowerblue', 
            markeredgecolor='black', ecolor='cornflowerblue')
ax1.plot(wv2, mmp, marker = 's', markerfacecolor = 'None', markeredgecolor = 'royalblue', 
        markersize = 8, ls='--', lw=2, alpha = 0.6, color='cornflowerblue', 
        label = rf'MCMC $\chi^2={chi_mc:.2f}$')
ax1.plot(wv2, smp, marker = 's', markerfacecolor = 'None', markeredgecolor = 'hotpink', 
        markersize = 8, ls='--', lw=2, alpha = 0.6, color='hotpink', 
        label = rf'SBI $\chi^2={chi_sbi:.2f}$')
ax1.grid(alpha=0.3, linestyle='--')
ax1.invert_yaxis()
ax1.legend()
ax1.set_ylabel('AB mag')

ax2.errorbar(wv, phot_mc['mag'] - rsgloader.dm - model_mag_mc, yerr=phot_mc['magerr'], linestyle='none', marker='o', markerfacecolor='cornflowerblue',
            markeredgecolor='black', ecolor='cornflowerblue', label = 'MCMC')
ax2.errorbar(wv, phot_mc['mag'] - rsgloader.dm - model_mag_sbi, yerr=phot_mc['magerr'], linestyle='none', marker='o', markerfacecolor='hotpink',
            markeredgecolor='black', ecolor='hotpink', label = r'SBI $\chi^2$')
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xlabel(r'$\lambda(\mu m)$')
ax2.set_ylabel('Residuals')
ax2.axhline(0.0, linestyle='--', alpha=0.5, color='black');

In [ ]:
t1 = pd.read_csv('../data/dolphot/ngc4449/sbi/ngc4449_infer.csv')
t1 = t1[t1['gal'] == 'ngc4449']
t2 = pd.read_csv('../data/sbi/ngc4449_infer_3.csv')

In [ ]:
plt.scatter(t1['teff_sbi'], t2['teff_sbi']*1000, s=0.1)
plt.plot([2500, 5000], [2500, 5000], color='black')

In [ ]:
newcat = pd.read_csv('../data/dolphot/ngc4449/sbi/ngc4449_infer.csv')
# newcat = pd.read_csv('../data/sbi/ngc4449_infer_3.csv')

# newcat.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
newcat = rsgcat[detm]
newcat[['teff_sbi', 'teff_sbi_elow', 'teff_sbi_eup', 'tdust_sbi', 'tdust_sbi_elow', 'tdust_sbi_eup', 
        'tau_sbi', 'tau_sbi_elow', 'tau_sbi_eup', 'lum_sbi', 'lum_sbi_elow', 'lum_sbi_eup', 
        'rv_sbi', 'rv_sbi_elow', 'rv_sbi_eup', 'av_sbi', 'av_sbi_elow', 'av_sbi_eup']] = np.nan
params_write = ['teff', 'tdust', 'tau', 'lum', 'rv', 'av']
device = 'cpu'

for idx in tqdm(newcat.index):
    col = newcat.loc[idx]
    obsmag = col[rsgloader.cols['magcols'][rsgloader.flt_mask]].to_numpy(dtype=float) - (rsgloader.gen_mc_obj.dm+30)
    obserr = col[rsgloader.cols['errcols'][rsgloader.flt_mask]].to_numpy(dtype=float)
    obserr = np.sqrt(obserr**2 + 0.01**2)
    obs = np.concatenate([obsmag, obserr])
    samp = sedfit.sample_with_timeout(hatp_x_y, sample_shape=(2500,), x=torch.as_tensor(obs.astype(np.float32)).to(device), timeout=1, show_progress_bars=False)
    if samp is not None:
        med = np.percentile(samp, 50, axis=0)
        newcat.loc[idx, list(map(lambda x: x + '_sbi', params_write))] = med
        newcat.loc[idx, list(map(lambda x: x + '_sbi_elow', params_write))] = med - np.percentile(samp, 16, axis=0)
        newcat.loc[idx, list(map(lambda x: x + '_sbi_eup', params_write))] = np.percentile(samp, 84, axis=0) - med

newcat.to_csv(os.path.join(rsgloader.procdir, 'sbi', f'{rsgloader.gal}_infer.csv'), index=False)

In [ ]:
newcat = newcat[~np.isnan(newcat['lum_sbi'])]
newcat = newcat[newcat['gal'] == 'ngc4449']

def lin(x, m, c): return m*x+c

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

d2 = (newcat[f1+'_mag'] > 10) & (newcat[f1+'_mag'] < 35) & (newcat[f2+'_mag'] > 10) & (newcat[f2+'_mag'] < 35)
rsg_cl, rsg_m = newcat[d2][f1+'_mag'] - newcat[d2][f2+'_mag'], newcat[d2][f2+'_mag']
sc = ax.scatter(rsg_cl, rsg_m, c=1000*newcat[d2]['teff_sbi'], cmap = 'viridis', s=.5)

plt.plot([0.0, 0.3], [24, 19], ls='--', color='orange')
plt.plot([0.165, 0.65], [24, 19], ls='-.', color='orange', label='literature cut')

cbar = fig.colorbar(sc, ax = ax, pad = 0.009)
cbar.ax.tick_params() 
ax.set_xlabel(rf'${f1} - {f2}$')
ax.set_ylabel(rf'${f2}$')
ax.set_title('NGC4449 + NGC4485 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-0.5, 2)
ax.set_ylim(24, 17)
# plt.show()

In [ ]:
newcat = newcat[~np.isnan(newcat['lum_sbi'])]

def lin(x, m, c): return m*x+c

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

d2 = (newcat[f1+'_mag'] > 10) & (newcat[f1+'_mag'] < 35) & (newcat[f2+'_mag'] > 10) & (newcat[f2+'_mag'] < 35)
rsg_cl, rsg_m = newcat[d2][f1+'_mag'] - newcat[d2][f2+'_mag'], newcat[d2][f2+'_mag']
sc = ax.scatter(rsg_cl, rsg_m, c=1000*newcat[d2]['teff_sbi'], cmap = 'viridis', s=.5)

plt.plot([0.0, 0.3], [24, 19], ls='--', color='orange')
plt.plot([0.165, 0.65], [24, 19], ls='-.', color='orange', label='literature cut')

cbar = fig.colorbar(sc, ax = ax, pad = 0.009)
cbar.ax.tick_params() 
ax.set_xlabel(rf'${f1} - {f2}$')
ax.set_ylabel(rf'${f2}$')
ax.set_title('NGC4449 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-0.5, 2)
ax.set_ylim(26, 20)
# plt.show()

In [ ]:
cl1 = newcat[d2]['F115W_mag'] - newcat[d2]['F200W_mag']
cm1 = newcat[d2]['F200W_mag']
l1 = (cm1 < 24) & (cm1 > lin(cl1, -16.667, 24))
l2 = (cm1 < 24) & (cm1 < lin(cl1, -10.31, 25.701))
l3 = (cm1 < 20) & (cm1 > lin(cl1, -10.31, 25.701))
lc = (l1&l2)|l3
rsglit = newcat[d2][lc]
agblit = newcat[d2][~lc]

In [ ]:
f = 1
plt.hist(f*rsglit['teff_sbi'], bins=51, histtype='step', label = 'RSG', lw=2, color='royalblue', density=True)
plt.hist(f*agblit['teff_sbi'], bins=51, histtype='step', label = 'AGB', lw=2, linestyle='--', color='hotpink', density=True)
plt.xlabel('Teff (K)')
plt.legend(); 

In [ ]:
# plt.scatter(np.log10(rsglit['teff_sbi']) + 3, rsglit['lum_sbi'], s=10, alpha=0.5, color='royalblue', label='RSG')
plt.scatter(np.log10(agblit['teff_sbi']), agblit['lum_sbi'], s=10, alpha=0.5, color='hotpink', label='AGB')
plt.scatter(np.log10(rsglit['teff_sbi']), rsglit['lum_sbi'], s=10, alpha=0.5, color='royalblue', label='RSG')
plt.gca().invert_xaxis()
plt.legend()
plt.xlabel('log(T)')
plt.ylabel('log(L)');

In [ ]:
def fmt(x):
    s = f"{x:.1f}"
    if s.endswith("0"):
        s = f"{x:.0f}"
    return rf"{s} \%" if plt.rcParams["text.usetex"] else f"{s} %"

fig, axs = plt.subplot_mosaic([['hist_teff', '.'],
                               ['contour', 'hist_lum']],
                              figsize=(9, 9),
                              width_ratios=(4, 1), height_ratios=(1, 4),
                              layout='tight', sharex=False, sharey=False, 
                              gridspec_kw={"hspace": 0.05, "wspace": 0.05})

axs['hist_teff'].sharex(axs['contour'])
axs['hist_lum'].sharey(axs['contour'])

rsg_logT = np.log10(rsglit['teff_sbi']) 
rsg_logL = rsglit['lum_sbi']

agb_logT = np.log10(agblit['teff_sbi']) 
agb_logL = agblit['lum_sbi']

xy_rsg = np.vstack([rsg_logT, rsg_logL])
kde_rsg = gaussian_kde(xy_rsg)
x_rsg, y_rsg = np.mgrid[rsg_logT.min():rsg_logT.max():100j, rsg_logL.min():rsg_logL.max():100j]
z_rsg = kde_rsg(np.vstack([x_rsg.ravel(), y_rsg.ravel()])).reshape(x_rsg.shape)
axs['contour'].scatter(rsg_logT, rsg_logL, s=5, marker='o', facecolor=None, edgecolor='royalblue', alpha=0.25, label='RSG')
rsg_cs = axs['contour'].contour(x_rsg, y_rsg, z_rsg, colors='royalblue', linewidths=1.5, linestyles='-', label='RSG', levels=np.linspace(0, 10, 10))
axs['contour'].clabel(rsg_cs, rsg_cs.levels[:3], fmt=fmt, fontsize=10, colors='black')

xy_agb = np.vstack([agb_logT, agb_logL])
kde_agb = gaussian_kde(xy_agb)
x_agb, y_agb = np.mgrid[agb_logT.min():agb_logT.max():100j, agb_logL.min():agb_logL.max():100j]
z_agb = kde_agb(np.vstack([x_agb.ravel(), y_agb.ravel()])).reshape(x_agb.shape)
axs['contour'].scatter(agb_logT, agb_logL, s=5, marker='x', facecolor=None, edgecolor='coral', alpha=0.25, label='AGB')
agb_cs = axs['contour'].contour(x_agb, y_agb, z_agb, colors='coral', linewidths=1.5, linestyles='--', label='AGB', levels=np.linspace(0, 10, 10))
axs['contour'].clabel(agb_cs, agb_cs.levels[:3], fmt=fmt, fontsize=10, colors='black')

axs['contour'].plot([3.5, 3.61], [5.5, 3.5], ls='--', color='black', alpha=0.5)

axs['contour'].set_xlabel(r'$\text{log}(T_{eff})$')
axs['contour'].set_ylabel(r'$\text{log}(L)$')
axs['contour'].legend()
axs['contour'].grid(alpha=0.3, linestyle='--')
axs['contour'].invert_xaxis()

axs['hist_teff'].hist(rsg_logT, bins=50, histtype='step', label = 'RSG', lw=2, color='royalblue', density=False)
axs['hist_teff'].hist(agb_logT, bins=50, histtype='step', label = 'AGB', lw=2, linestyle='--', color='coral', density=False)
axs['hist_teff'].tick_params(axis='x', which='both', labelbottom=False)

axs['hist_lum'].hist(rsg_logL, bins=30, histtype='step', label = 'RSG', lw=2, color='royalblue', density=False, orientation='horizontal')
axs['hist_lum'].hist(agb_logL, bins=30, histtype='step', label = 'AGB', lw=2, linestyle='--', color='coral', density=False, orientation='horizontal')
axs['hist_lum'].tick_params(axis='y', which='both', labelleft=False)

plt.show()

In [ ]:
pdf_rsg = kde_rsg.pdf(xy_rsg)
pdf_agb = kde_agb.pdf(xy_rsg)
rsgm = pdf_rsg > pdf_agb
rsglit[rsgm].to_csv('data/sbi/kde_rsg.csv', index=False)

In [ ]:
newcat_av = newcat.sort_values(by='av_sbi', ascending=True)
m = newcat_av['Dec'] < 43
plt.scatter(newcat_av['RA'][m], newcat_av['Dec'][m], s=newcat_av['av_sbi'][m], c=newcat_av['av_sbi'][m])
plt.colorbar();

### base catalog

In [ ]:
mc_obj = mcmc(dm=30.57, dmerr=0.06, z=-0.25)
mc_obj.verbose = True
mc_obj.bounds['luminosity'] = [10**3.5, 10**6.0]
mc_obj.bounds['temperature'] = [3500., 5000.]

sample_models = np.zeros((10000, len(nrc_filts)))
sample_params = mc_obj.get_init_pos(6, 10000)
sample_params = sample_params[sample_params[:, 0].argsort()]
for i, p_ in enumerate(sample_params):
    model_mag = np.array([mc_obj.model[f](sample_params[i]).flatten()[0] for f in nrc_filts]) + mc_obj.dm
    sample_models[i, :] = model_mag

In [ ]:
mc_obj = mcmc(dm=30.57, dmerr=0.06, z=-0.25)
mc_obj.verbose = True
mc_obj.bounds['luminosity'] = [10**3.5, 10**3.6]

base_models = np.zeros((2000, len(nrc_filts)))
basedf_cols = [i+'_mag' for i in nrc_filts]

sample_params = mc_obj.get_init_pos(6, 1000)
for i, p_ in enumerate(sample_params):
    model_mag = np.array([mc_obj.model[f](sample_params[i]).flatten()[0] for f in nrc_filts]) + mc_obj.dm
    base_models[i, :] = model_mag


mc_obj.bounds['luminosity'] = [10**5.8, 10**6]
sample_params = mc_obj.get_init_pos(6, 1000)
for i, p_ in enumerate(sample_params):
    model_mag = np.array([mc_obj.model[f](sample_params[i]).flatten()[0] for f in nrc_filts]) + mc_obj.dm
    base_models[i+1000, :] = model_mag

min_model = np.max(base_models[base_models[:, 1] < 26.20], axis=0) #TRGB at F090W 26.20 mag for NGC5643
min_model_df = pd.DataFrame([dict(zip(basedf_cols, min_model))])

max_model = np.min(base_models[base_models[:, 1] < 26.20], axis=0)
max_model_df = pd.DataFrame([dict(zip(basedf_cols, max_model))])

In [ ]:
fig, ax = plt.subplots(1, 1)
for i, m_ in enumerate(sample_models):
    ax.plot(wv_all, m_, color='gray', alpha=0.02)
ax.plot(wv_all, min_model, color='black')
ax.plot(wv_all, max_model, color='black')
ax.invert_yaxis()
ax.grid(alpha=0.2, linestyle='--')
ax.set_xlabel(r'$\lambda(\mu m)$')
ax.set_ylabel('AB mag');

In [ ]:
fig, ax = plt.subplots(1, 1)
for i, m_ in enumerate(base_models[base_models[:, 1] < 26.20]):
    ax.plot(wv_all, m_, color='gray', alpha=0.02)
ax.plot(wv_all, min_model, color='black')
ax.plot(wv_all, max_model, color='black')
ax.invert_yaxis()
ax.grid(alpha=0.2, linestyle='--')
ax.set_xlabel(r'$\lambda(\mu m)$')
ax.set_ylabel('AB mag')

In [ ]:
def detmask(cat_mags_det, min_det=4):
    cat_wv = np.array([float(i[1:4])/100 for i in cat_mags_det.columns])

    opt_dets = cat_mags_det[cat_mags_det.columns[(cat_wv < 1.2)]]
    nir_dets = cat_mags_det[cat_mags_det.columns[(cat_wv > 1.2) & (cat_wv < 2.6)]]
    mir_dets = cat_mags_det[cat_mags_det.columns[(cat_wv > 2.6)]]

    ndetm = cat_mags_det.sum(axis=1) > min_det
    detm = (opt_dets.sum(axis=1) > 0) & (nir_dets.sum(axis=1) > 1) & (mir_dets.sum(axis=1) > 1) & ndetm

    return detm

def peakmask(cat_mags):
    cat_wv = np.array([float(i[1:4])/100 for i in cat_mags.columns])

    opt_dets = cat_mags[cat_mags.columns[(cat_wv < 1.2)]]
    nir_dets = cat_mags[cat_mags.columns[(cat_wv > 1.2) & (cat_wv < 2.6)]]
    mir_dets = cat_mags[cat_mags.columns[(cat_wv > 2.6)]]

    opt_peak, opt_med = opt_dets.min(axis=1), opt_dets.median(axis=1)
    nir_peak, nir_med = nir_dets.min(axis=1), nir_dets.median(axis=1)
    mir_peak, mir_med = mir_dets.min(axis=1), mir_dets.median(axis=1)

    req_nir_peak = (nir_peak < opt_peak) & (nir_peak < mir_peak)
    dust_peak = (nir_peak < opt_peak) & ((mir_med - opt_med) < -0.5) & ((nir_med - mir_med) < 1.0) & ((nir_med - mir_med) > 0)
    peakm = req_nir_peak | dust_peak
    print(req_nir_peak.sum(), dust_peak.sum())
    
    return peakm

def colormask(cat_mags, fls, sample_models, min_width=0.5, rel_err=0.25):
    cat_mags = cat_mags[fls]
    color_dict = {}
    cmb_iterator = list(itertools.combinations(nrc_filts, 2))
    idx_iterator = list(itertools.combinations(range(0, 29), 2))
    for i, j in zip(idx_iterator, cmb_iterator):
        f1, f2 = j[0], j[1]
        i1, i2 = i[0], i[1]
        index = f1+'_'+f2
        color_bounds = sample_models[:, i1] - sample_models[:, i2]
        cl_min = min(color_bounds) - rel_err #*np.abs(min(color_bounds))
        cl_max = max(color_bounds) + rel_err #*np.abs(max(color_bounds))
        cld = cl_max - cl_min
        if cld < min_width:
            cl_min = cl_min - (min_width - cld)/2
            cl_max = cl_max + (min_width - cld)/2
        color_dict[index] = (cl_min, cl_max)

    nwm = np.array(['N' in i for i in fls]) | np.array(['300M' in i for i in fls])
    mfls = fls[~nwm]

    colorm = np.array([True]*len(cat_mags))
    # colorm = np.zeros(len(cat_mags))
    for i, j in itertools.combinations(mfls, 2):
        idx_ = i.replace('_mag', '')+'_'+j.replace('_mag', '')
        cat_color = cat_mags[i] - cat_mags[j]
        dm = (cat_mags[i] > 90) | (cat_mags[j] > 90)
        colm_ = (cat_color > color_dict[idx_][0]) & (cat_color < color_dict[idx_][1])
        colm_ = colm_ | dm
        colorm = colorm & colm_
        # colorm = colorm + colm_.astype(int)
    # colorm = colorm > ((len(mfls)-1) * (len(mfls) - 2))/2 + 5
    return colorm, color_dict

fls = ['mag' in i for i in cat.columns]
fls = cat.columns[fls]
cat_mags = cat[fls]
cat_wv = np.array([float(i[1:4])/100 for i in cat_mags.columns])
cat_mags = cat_mags.replace(99.999, 0.5)
cat_mags = cat_mags.replace(np.nan, 0.5)

mindf = cat_mags-min_model_df[fls].to_numpy()
cat_mags = cat_mags.replace(0.5, 99.999)
maxdf = cat_mags-max_model_df[fls].to_numpy()
difm = (mindf < 0).all(axis=1) & (maxdf > 0).all(axis=1)
ndet = (cat_mags > 10) & (cat_mags < 38) 
ndetm = detmask(ndet)
base_mask = difm & ndetm  
rsgcat = cat[base_mask] #.replace(np.nan, 99.999)
print(len(rsgcat))

# f1, f2 = 'F115W', 'F200W'
# jkm = (rsgcat[f1+'_mag'] < 90) & (rsgcat[f2+'_mag'] < 90)
# cl, m = rsgcat[f1+'_mag'] - rsgcat[f2+'_mag'], rsgcat[f2+'_mag']
# # rsgmask = (m<26) & (m > lin(cl, -20.0, 33.0-3)) & (m < lin(cl, -8.0, 25+3.5))
# rsgmask = (m < 26) & (cl > 0.1) & jkm
# agbmask = (m < 26) & ((m > lin(cl, -10.0, 25+6)) | (m < lin(cl, -20.0, 33.0-5)))
# agbmask = agbmask & jkm
# rsgcat = rsgcat[rsgmask].replace(np.nan, 99.999)
# print(len(rsgcat))

# colorm, color_dict = colormask(rsgcat, fls, sample_models, min_width=0.2, rel_err=0.3)

# peakm = peakmask(rsgcat)
# rsgcat = rsgcat[colorm]
# print(len(rsgcat))
# rsgcat = rsgcat[~colorm][fls].replace(99.999, np.nan)

# f1, f2 = 'F115W', 'F200W'
# cl, m = rsgcat[f1+'_mag'] - rsgcat[f2+'_mag'], rsgcat[f2+'_mag']
# rsgmask = (m<26) & (m > lin(cl, -20.0, 33.0-3)) & (m < lin(cl, -8.0, 25+3.5))
# rsgcat = rsgcat[~rsgmask]

flts = [i.replace('_mag','') for i in fls]
magcols = [i+'_mag' for i in flts]
errcols = [i+'_err' for i in flts]

In [ ]:
min_width, rel_err = 1.0, 1.0

cat_mags = cat_mags[fls]
nwm = np.array(['N' in i for i in fls]) #| np.array(['M' in i for i in fls])
mfls = fls[~nwm]

colorm = np.array([True]*len(cat_mags))
for i, j in itertools.combinations(mfls, 2):
    idx_ = i.replace('_mag', '')+'_'+j.replace('_mag', '')
    cat_color = testcol[i] - testcol[j]
    dm = (testcol[i] > 90) | (testcol[j] > 90)
    colm_ = (cat_color > color_dict[idx_][0]) & (cat_color < color_dict[idx_][1])
    colm_ = colm_ | dm
    if not colm_:
        print(idx_, cat_color, color_dict[idx_], dm)
    colorm = colorm & colm_

In [ ]:
plt.scatter(cat_wv, rsgcat.sample(1).values[0])
plt.plot(wv_all, min_model, color='black')
plt.plot(wv_all, max_model, color='black')
plt.gca().invert_yaxis()
plt.grid(alpha=0.2, linestyle='--')

In [ ]:
teff_ = np.arange(2600, 5050, 50)
tdust_ = np.arange(200, 1800, 100)
tau_ = np.array(list(np.linspace(0.01, 2, 21)) + list(np.arange(2.5, 5.5, 0.5)))
Av_ = np.array(list(np.linspace(0, 1, 5))+ list(np.linspace(1.5, 3, 4)))
len(teff_) * len(tdust_) * len(tau_) * len(Av_)

In [ ]:
modeldf = pd.DataFrame(columns = ['Teff', 'Tdust', 'Tau', 'Av']+nrc_filts)
for a1 in tqdm(teff_):
    for a2 in tdust_:
        for a3 in tau_:
            for a4 in Av_:
                pm_ = [a1, a2, a3, 1e3, 3.1, a4]
                model_mag = np.array([mc_obj.model[f](pm_).flatten()[0] for f in nrc_filts]) + mc_obj.dm
                modeldf.loc[len(modeldf), ['Teff', 'Tdust', 'Tau', 'Av']+nrc_filts] = [a1, a2, a3, a4] + list(model_mag)
modeldf.to_csv('modeldf_ngc5643.csv', index=False)

In [ ]:
modeldf = pd.read_csv('data/dolphot/ngc5643/ngc5643_modeldf.csv')

In [ ]:
def chimin(phot, modeldf):
    modelmagval = modeldf[phot['inst_filt']].values
    submagval = modelmagval - phot['mag']
    meanval = np.average(submagval, weights = 1/phot['magerr']**2, axis=1)
    submagval = submagval - meanval[:, None]
    chisqe = np.sum(submagval**2/phot['magerr']**2, axis=1)/len(phot['mag'] - 1)
    chisq = np.sum(submagval**2, axis=1)/len(phot['mag'] - 1)
    minchisq = np.argmin(chisqe)

    # print(chisq[minchisq])
    # # otherbestmodel = modeldf.loc[55544][phot['inst_filt']].values - np.log10(21729/1e3)*2.5
    # otherbestmodel = chi_mag
    # photmag=phot['mag']
    # print(np.sum((otherbestmodel - photmag)**2)/(len(photmag) - 1))
    # print((otherbestmodel - photmag)**2)
    # print(submagval[minchisq]**2)

    bestparams = modeldf.loc[minchisq, ['Teff', 'Tdust', 'Tau', 'Av']].values
    pm_ = [bestparams[0], bestparams[1], bestparams[2], 1e3, 3.1, bestparams[3]]
    bestmodel = np.array([mc_obj.model[f](pm_).flatten()[0] for f in phot['inst_filt']]) + mc_obj.dm
    lum = 10**(np.mean((phot['mag'] - bestmodel)/-2.5)) * 1e3
    return chisq[minchisq], minchisq, lum, submagval[minchisq]

In [ ]:
c_, m_, l_, res = chimin(phot, modeldf)
print(c_, l_)
print(modeldf.loc[m_, ['Teff', 'Tdust', 'Tau', 'Av']])

In [ ]:
plt.plot(wv_all, modeldf.loc[m_][nrc_filts].values - np.log10(l_/1e3)*2.5, color='coral', ls='--', marker='s', markerfacecolor='none', markersize=8)
plt.plot(wv_all, chi_mag_plot, color='green', ls='--', marker='s', markerfacecolor='none', markersize=8)
# plt.plot(wv_all, modeldf.loc[17065][nrc_filts].values - np.log10(7479/1e3)*2.5, color='green', ls='--', marker='s', markerfacecolor='none', markersize=8)
plt.scatter(wv, phot['mag'], fc='cornflowerblue', ec='black')
# plt.scatter(wv_all, np.array([mc_obj.model[f]([3800, 1300, 0.1095, 56329, 3.1, 0.75]).flatten()[0] for f in nrc_filts]) + mc_obj.dm, color='blue')
plt.gca().invert_yaxis()

In [ ]:
rescat = pd.DataFrame(columns=sedfit.cols['flts'])

In [ ]:
keep_narrow = True
rsgcat[['chimin', 'teff', 'tdust', 'tau', 'av', 'lum']] = np.nan
for idx in tqdm(rsgcat.index):
    testcol = rsgcat.loc[idx]

    phot = {'mag': testcol[sedfit.cols['magcols']].values,
            'magerr': testcol[sedfit.cols['errcols']].values,
            'inst_filt': np.array(sedfit.cols['flts']),
            'index': f'ngc5643_{int(testcol['index'])}'}

    if not keep_narrow:
            fl_mask = np.array(['N' in i for i in phot['inst_filt']]) | np.array(['300M' in i for i in phot['inst_filt']])
            phot['mag'] = phot['mag'][~fl_mask]
            
            phot['magerr'] = phot['magerr'][~fl_mask]
            phot['inst_filt'] = phot['inst_filt'][~fl_mask]

    limmask = (phot['mag'] > 90.0) | (phot['magerr'] > 90.0) | (np.isnan(phot['mag'])) | (np.isnan(phot['magerr']))
    phot['mag'] = phot['mag'][~limmask]
    phot['magerr'] = phot['magerr'][~limmask]
    phot['inst_filt'] = phot['inst_filt'][~limmask]
    c_, m_, l_, res = chimin(phot, modeldf)
    rescat.loc[len(rescat), phot['inst_filt']] = res
#     rsgcat.loc[idx, ['chimin', 'lum']] = c_, l_
#     rsgcat.loc[idx, ['teff', 'tdust', 'tau', 'av']] = modeldf.loc[m_, ['Teff', 'Tdust', 'Tau', 'Av']].values

In [ ]:
fig, ax = plt.subplots()
resvals = []
for col in rescat.columns:
    nanmask = rescat[col].isna()
    resvals.append(rescat[col][~nanmask].values)
    print(col, np.std(rescat[col][~nanmask].values, ddof=1))
labels = sedfit.cols['flts']

bplot = ax.boxplot(resvals,
                   patch_artist=True,  # fill with color
                   tick_labels=labels)
ax.set_xticklabels(labels=labels, rotation = 90);

In [ ]:
rescat.to_csv('data/dolphot/ngc5643/rescat.csv', index=False)

### RSG-AGB split

In [ ]:
sedfit.cat = sedfit.read_cat(sedfit.photfile_path)
cat = sedfit.read_cat(sedfit.photfile_path)

In [ ]:
f1, f2 = 'F090W', 'F150W'
det_jk = (sedfit.cat[f'{f1}_mag'] > 15) & (sedfit.cat[f'{f1}_mag'] < 35) & (sedfit.cat[f'{f2}_mag'] > 15) & (sedfit.cat[f'{f2}_mag'] < 35)
plt.hist2d(sedfit.cat[det_jk][f'{f1}_mag'] - sedfit.cat[det_jk][f'{f2}_mag'], sedfit.cat[det_jk][f'{f1}_mag'], bins=1400, cmap = 'inferno', norm='log');
# plt.scatter(sedfit.cat[det_jk][f'{f1}_mag'] - sedfit.cat[det_jk][f'{f2}_mag'] + 26.28 - 28.02 - 25.90 + 28.02, sedfit.cat[det_jk][f'{f1}_mag'] + 26.28 - 28.02, s=0.001, color='orange', alpha=0.1);
# plt.scatter(sedfit.cat[det_jk][f'{f1}_mag'] - sedfit.cat[det_jk][f'{f2}_mag'], sedfit.cat[det_jk][f'{f1}_mag'], s=0.001, color='orange', alpha=0.1);
plt.axhline(26.19, ls='--', color='black')
plt.xlabel(f'{f1} - {f2}')
plt.ylabel(f'{f1}')
plt.xlim(0.5, 2.5)
plt.ylim(24.5, 29.5)
plt.gca().invert_yaxis()

In [ ]:
def lin(x, m, c): return m*x+c

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

det_jk = (cat[f1+'_mag'] > 18) & (cat[f1+'_mag'] < 32) & (cat[f2+'_mag'] > 18) & (cat[f2+'_mag'] < 32)
cl, m = cat[det_jk][f1+'_mag'] - cat[det_jk][f2+'_mag'], cat[det_jk][f2+'_mag']
rsg_det_jk = (rsgcat[f1+'_mag'] > 18) & (rsgcat[f1+'_mag'] < 32) & (rsgcat[f2+'_mag'] > 18) & (rsgcat[f2+'_mag'] < 32)
rsg_cl, rsg_m = rsgcat[rsg_det_jk][f1+'_mag'] - rsgcat[rsg_det_jk][f2+'_mag'], rsgcat[rsg_det_jk][f2+'_mag']

ax.hist2d(cl, m, bins = 400, norm = 'log', cmap = 'Grays')
hist = ax.hist2d(rsg_cl, rsg_m, bins = 200, norm = 'log', cmap = 'magma')

ax.axhline(26, color = 'red', label='TRGB', linestyle = '-.')
ax.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.22625, 1, 100), lin(np.linspace(0.22625, 1, 100), -10.0, 25+4.5), linestyle='-.', color = 'orange')
# ax.scatter(phot['mag'][1] - phot['mag'][3], phot['mag'][3], marker='*', s=150, fc='orange', ec='black')

ax.axhline(23, color = 'red', linestyle = '--')
ax.axhline(23.2, color = 'red', linestyle = '--')

cbar = fig.colorbar(hist[3], ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-2, 2.5)
ax.set_ylim(32.5, 17)
plt.show()

In [ ]:
binm = (m > 23.2) & (m < 23.9) & (cl < 5)
cl_binm = cl[binm]
plt.hist(cl_binm, bins=300, histtype='step', density=True);
plt.axvline(0.52, ls='--', lw=0.5, color='black')
# plt.yscale('log')

In [ ]:
def lin(x, m, c): return m*x+c

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag']
# rsg_cl, rsg_m = rsgcat[f1+'_mag'] - rsgcat[f2+'_mag'], rsgcat[f2+'_mag']
# mask = (cl > 0.5) & (cl < 1.2) & (m < 26)
# cl, m = cl[mask], m[mask]

hist = ax.hist2d(cl, m, bins = 5000, norm = 'log', cmap = 'magma')
# hist = ax.hist2d(rsg_cl, rsg_m, bins = 4000, norm = 'log', cmap = 'magma')
# plt.scatter(cl, m, s=0.1, alpha=0.9)

# ax.axhline(26, color = 'red', label='TRGB', linestyle = '-.')
ax.plot(np.linspace(0.1905, 1.5, 100), lin(np.linspace(0.1905, 1.5, 100) + 0.44, -15.366, 22.048-17.41+30.57), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.22625, 2.0, 100), lin(np.linspace(0.22625, 2.0, 100) + 0.44, -11.618, 21.917-17.41+30.57), linestyle='-.', color = 'orange')
ax.plot(np.linspace(0.22625, 2.0, 100), lin(np.linspace(0.22625, 2.0, 100) + 0.44, -9.268, 22.481-17.41+30.57), linestyle='-.', color = 'orange')

ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
# ax.set_xlim(0.5, 1.2)
# ax.set_ylim(26, 17)
ax.set_xlim(-2, 3)
ax.set_ylim(32.5, 17)
plt.show()

In [ ]:
plt.hist(rsgcat['chimin'], bins=40);
plt.yscale('log')

In [ ]:
rsgcat = pd.read_csv('data/dolphot/ngc5643/ngc5643_sil_rsgcat.csv')
rsgcat = rsgcat[rsgcat['teff_chisq'] < 4700]

In [ ]:
rsgcat

In [ ]:
chi_all = rsgcat['chimin'].values

In [ ]:
plt.hist(chi_all, bins=100)
plt.yscale('log')
plt.xlim(0.0, 0.2)
sig = np.std(chi_all, ddof=1)
me = np.median(chi_all)
plt.axvline(sig, ls='--', color='black')
plt.axvline(me, ls='--', color='black')

In [ ]:
p_ = np.arange(10, 100, 10)
pval = [np.percentile(chi_all, i) for i in p_]
plt.plot(p_, pval, marker='o')

In [ ]:
np.percentile(chi_all, 84)

In [ ]:
plt.hist(rsgcat['tau_chisq'], bins = 25);

In [ ]:
plt.hist(np.log10(rsgcat['lum_chisq']), bins = 25);

In [ ]:
cm = rsgcat['chimin'] < np.percentile(rsgcat['chimin'], 50) #0.008
tm = (rsgcat['teff_chisq'] > 2600) & (rsgcat['teff_chisq'] < 4700)
tum = (rsgcat['tau_chisq'].values > 1) & (rsgcat['teff_chisq'] > 3000)
lm = (np.log10(rsgcat['lum_chisq'].values) > 4.0) & (rsgcat['teff_chisq'] < 4500)
netmask = cm & (tm | tum | lm) 
# netmask = cm & tm
netmask.sum()

In [ ]:
(cm & tm).sum()

In [ ]:
def lin(x, m, c): return m*x+c

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

# cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag']
rsg_cl, rsg_m = rsgcat[netmask][f1+'_mag'] - rsgcat[netmask][f2+'_mag'], rsgcat[netmask][f2+'_mag']

# ax.hist2d(cl, m, bins = 4000, norm = 'log', cmap = 'Grays')
sc = ax.scatter(rsg_cl, rsg_m, c=np.log10(rsgcat[netmask]['teff_chisq']), cmap = 'viridis', s=.5)

ax.axhline(26, color = 'red', label='TRGB', linestyle = '-.')
ax.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.35, 1, 100), lin(np.linspace(0.35, 1, 100), -10.0, 25+4.5), linestyle='-.', color = 'orange')

cbar = fig.colorbar(sc, ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-2, 2.5)
# ax.set_xlim(0, 1)
ax.set_ylim(32.5, 17)
# ax.set_ylim(26, 20)
plt.show()

In [ ]:
m1 = (rsgcat[netmask]['F115W_mag'] > 10) & (rsgcat[netmask]['F115W_mag'] < 32)  & (rsgcat[netmask]['F200W_mag'] > 10) & (rsgcat[netmask]['F200W_mag'] < 32)
cl1 = rsgcat[netmask][m1]['F115W_mag'] - rsgcat[netmask][m1]['F200W_mag']
cm1 = rsgcat[netmask][m1]['F200W_mag']
m2 = cm1 > lin(cl1, -20.0, 33.0-3)
m3 = cm1 < lin(cl1, -10.0, 25+4.5)

rsg_lit = rsgcat[netmask][m1][m2&m3]
csm1 = (rsg_lit['chimin'] < np.percentile(rsgcat[netmask]['chimin'], 50)) #& (np.log10(rsg_lit['lum_chisq']) > 4.2) # & (rsg_lit['teff_chisq'] == 4200)
agb_lit = rsgcat[netmask][m1][m2&~m3]
csm2 = (agb_lit['chimin'] < np.percentile(rsgcat[netmask]['chimin'], 50)) #& (np.log10(agb_lit['lum_chisq']) > 4.2) # & (agb_lit['teff_chisq'] == 4200)
plt.hist(rsg_lit[csm1]['teff_chisq'], bins=50, histtype='step', label = 'RSG', lw=2, color='royalblue', density=True)
plt.hist(agb_lit[csm2]['teff_chisq'], bins=50, histtype='step', label = 'AGB', lw=2, linestyle='--', color='hotpink', density=True)
# plt.ylim(0, 2000)
plt.xlabel('Teff (K)')
plt.legend(); 
# plt.savefig('plots/ngc543_rsg_line_lc.png', dpi=300, bbox_inches='tight');

In [ ]:
plt.scatter(np.log10(rsg_lit['teff_chisq']), np.log10(rsg_lit['lum_chisq']), s=10, alpha=0.5, color='royalblue', label='RSG')
plt.scatter(np.log10(agb_lit['teff_chisq']), np.log10(agb_lit['lum_chisq']), s=10, alpha=0.5, color='hotpink', label='AGB')
plt.gca().invert_xaxis()
plt.legend()
# plt.xlim(6, 3)
plt.xlabel('log(T)')
plt.ylabel('log(L)');

In [ ]:
def fmt(x):
    s = f"{x:.1f}"
    if s.endswith("0"):
        s = f"{x:.0f}"
    return rf"{s} \%" if plt.rcParams["text.usetex"] else f"{s} %"

fig, axs = plt.subplot_mosaic([['hist_teff', '.'],
                               ['contour', 'hist_lum']],
                              figsize=(9, 9),
                              width_ratios=(4, 1), height_ratios=(1, 4),
                              layout='tight', sharex=False, sharey=False, 
                              gridspec_kw={"hspace": 0.05, "wspace": 0.05})

axs['hist_teff'].sharex(axs['contour'])
axs['hist_lum'].sharey(axs['contour'])

rsg_logT = np.log10(rsg_lit['teff_chisq'])
rsg_logL = np.log10(rsg_lit['lum_chisq'])

agb_logT = np.log10(agb_lit['teff_chisq'])
agb_logL = np.log10(agb_lit['lum_chisq'])

xy_rsg = np.vstack([rsg_logT, rsg_logL])
kde_rsg = gaussian_kde(xy_rsg)
x_rsg, y_rsg = np.mgrid[rsg_logT.min():rsg_logT.max():100j, rsg_logL.min():rsg_logL.max():100j]
z_rsg = kde_rsg(np.vstack([x_rsg.ravel(), y_rsg.ravel()])).reshape(x_rsg.shape)
axs['contour'].scatter(rsg_logT, rsg_logL, s=5, marker='o', facecolor=None, edgecolor='royalblue', alpha=0.25, label='RSG')
rsg_cs = axs['contour'].contour(x_rsg, y_rsg, z_rsg, colors='royalblue', linewidths=1.5, linestyles='-', label='RSG', levels=np.linspace(0, 10, 6))
# axs['contour'].clabel(rsg_cs, rsg_cs.levels, fmt=fmt, fontsize=10, colors='black')

xy_agb = np.vstack([agb_logT, agb_logL])
kde_agb = gaussian_kde(xy_agb)
x_agb, y_agb = np.mgrid[agb_logT.min():agb_logT.max():100j, agb_logL.min():agb_logL.max():100j]
z_agb = kde_agb(np.vstack([x_agb.ravel(), y_agb.ravel()])).reshape(x_agb.shape)
axs['contour'].scatter(agb_logT, agb_logL, s=5, marker='x', facecolor=None, edgecolor='coral', alpha=0.25, label='AGB')
agb_cs = axs['contour'].contour(x_agb, y_agb, z_agb, colors='coral', linewidths=1.5, linestyles='--', label='AGB', levels=np.linspace(0, 10, 6))
# axs['contour'].clabel(agb_cs, agb_cs.levels, fmt=fmt, fontsize=10, colors='black')

xs = np.linspace(3.45, 3.57, 100)
ys = lin(xs, -16.67, 63)
axs['contour'].plot(xs, ys, ls='--', color='black')

axs['contour'].set_xlabel(r'$\text{log}(T_{eff})$')
axs['contour'].set_ylabel(r'$\text{log}(L)$')
axs['contour'].legend()
axs['contour'].grid(alpha=0.3, linestyle='--')
axs['contour'].invert_xaxis()

axs['hist_teff'].hist(rsg_logT, bins=50, histtype='step', label = 'RSG', lw=2, color='royalblue', density=False)
axs['hist_teff'].hist(agb_logT, bins=50, histtype='step', label = 'AGB', lw=2, linestyle='--', color='coral', density=False)
axs['hist_teff'].tick_params(axis='x', which='both', labelbottom=False)

axs['hist_lum'].hist(rsg_logL, bins=30, histtype='step', label = 'RSG', lw=2, color='royalblue', density=False, orientation='horizontal')
axs['hist_lum'].hist(agb_logL, bins=30, histtype='step', label = 'AGB', lw=2, linestyle='--', color='coral', density=False, orientation='horizontal')
axs['hist_lum'].tick_params(axis='y', which='both', labelleft=False)

plt.show()

In [ ]:
full_logT, full_logL = np.log10(rsgcat[netmask]['teff_chisq']), np.log10(rsgcat[netmask]['lum_chisq'])
hr_mask = full_logL > lin(full_logT, -16.67, 63)
rsg_hr = rsgcat[netmask][hr_mask]
agb_hr = rsgcat[netmask][~hr_mask]

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag']
rsg_cl, rsg_m = rsg_hr[f1+'_mag'] - rsg_hr[f2+'_mag'], rsg_hr[f2+'_mag']
agb_cl, agb_m = agb_hr[f1+'_mag'] - agb_hr[f2+'_mag'], agb_hr[f2+'_mag']

ax.hist2d(cl, m, bins = 4000, norm = 'log', cmap = 'Grays')
ax.scatter(agb_cl, agb_m, color='orange', alpha=0.8, s=1, label='AGB (HR selected)')
ax.scatter(rsg_cl, rsg_m, color='cornflowerblue', alpha=0.8, s=1, label='RSG (HR selected)')
# ax.scatter(agb_cl, agb_m, color='orange', alpha=0.8, s=1, label='AGB (HR selected)')

ax.axhline(26, color = 'red', label='TRGB', linestyle = '-.')
ax.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.35, 1, 100), lin(np.linspace(0.35, 1, 100), -10.0, 25+4.5), linestyle='-.', color = 'crimson')

ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-2, 2.5)
# ax.set_xlim(0, 1)
ax.set_ylim(32, 17)
# ax.set_ylim(26, 20)
plt.show()

In [ ]:
(np.log10(rsg_hr['lum_chisq']) > 4.2).sum(), len(rsg_hr)

In [ ]:
plt.hist(np.log10(rsg_hr['lum_chisq']), bins=40);

In [ ]:
plt.hist(rsg_hr['teff_chisq'], bins=35, histtype='step', label = 'RSG', lw=2, color='royalblue', density=False)
plt.hist(agb_hr['teff_chisq'], bins=20, histtype='step', label = 'AGB', lw=2, linestyle='--', color='hotpink', density=False)
# plt.ylim(0, 2000)
plt.xlabel('Teff (K)')
plt.legend();
# plt.savefig('plots/ngc543_rsg_line_lc.png', dpi=300, bbox_inches='tight');

In [ ]:
de_rsg = (rsg_hr['tau_chisq'] > 3)
plt.scatter(np.log10(rsg_hr['teff_chisq']), np.log10(rsg_hr['lum_chisq']), s=10, alpha=0.5, color='royalblue', label='RSG')
plt.scatter(np.log10(agb_hr['teff_chisq']), np.log10(agb_hr['lum_chisq']), s=10, alpha=0.5, color='hotpink', label='AGB')
plt.scatter(np.log10(rsg_hr[de_rsg]['teff_chisq']), np.log10(rsg_hr[de_rsg]['lum_chisq']), s=40, marker='^',
            alpha=0.9, fc='violet', ec='black', label='DE-RSG')
plt.gca().invert_xaxis()
plt.legend()
# plt.xlim(6, 3)
plt.xlabel('log(T)')
plt.ylabel('log(L)');

In [ ]:
jk = (rsg_hr['F115W_mag'] - rsg_hr['F200W_mag']).values
k = rsg_hr['F200W_mag']
dm = (jk > -10) & (jk < 10) & (k > 10) & (k < 32)
lm = k > lin(jk, -10.0, 25+4.5)

plt.hist(rsg_hr['tau_chisq'][dm&~lm], bins=20, histtype='step', color='orange', label='Inside line cut', linestyle='-', lw=2, density=True)
plt.hist(rsg_hr['tau_chisq'][dm&lm], bins=20, histtype='step', color='red', label='Outside line cut', linestyle='--', lw=2, density=True)
plt.xlabel(r'$\tau$')
plt.legend();

In [ ]:
rls = rsg_lit[csm1].replace(99.999, np.nan).sample(1).iloc[0]
als = agb_lit[csm2].replace(99.999, np.nan).sample(1).iloc[0]
nwm = np.array(['N' in i for i in sedfit.cols['magcols']])
v1, v2 = np.nanmin(rls[sedfit.cols['magcols']].values[~nwm]), np.nanmin(als[sedfit.cols['magcols']].values[~nwm])
plt.plot(sedfit.cols['cat_wv'][~nwm], rls[sedfit.cols['magcols']].values[~nwm]-v1, marker='o', markersize=8, markeredgecolor='black', color='royalblue', label=f'RSG ({rls['teff_chisq']} K)', alpha=0.75)
plt.plot(sedfit.cols['cat_wv'][~nwm], als[sedfit.cols['magcols']].values[~nwm]-v2, marker='o', markersize=8, markeredgecolor='black', color='hotpink', label=f'AGB ({als['teff_chisq']} K)', alpha=0.75)
plt.legend()
plt.xlabel(r'$\lambda\,(\mu m)$')
plt.gca().invert_yaxis()

In [ ]:
rls = rsg_hr.replace(99.999, np.nan).sample(1).iloc[0]
als = agb_hr.replace(99.999, np.nan).sample(1).iloc[0]
nwm = np.array(['N' in i for i in sedfit.cols['magcols']])
v1, v2 = np.nanmin(rls[sedfit.cols['magcols']].values[~nwm]), np.nanmin(als[sedfit.cols['magcols']].values[~nwm])
plt.plot(sedfit.cols['cat_wv'][~nwm], rls[sedfit.cols['magcols']].values[~nwm]-v1, marker='o', markersize=8, markeredgecolor='black', color='royalblue', label=f'RSG ({rls['teff_chisq']} K)', alpha=0.75)
plt.plot(sedfit.cols['cat_wv'][~nwm], als[sedfit.cols['magcols']].values[~nwm]-v2, marker='o', markersize=8, markeredgecolor='black', color='hotpink', label=f'AGB ({als['teff_chisq']} K)', alpha=0.75)
plt.legend()
plt.xlabel(r'$\lambda\,(\mu m)$')
plt.gca().invert_yaxis()

In [ ]:
plt.hist(rsgcat[netmask]['teff_chisq'], bins=50)
plt.axvline(3400, ls='--', color='black')
plt.axvline(4000, ls='--', color='black');

In [ ]:
from sklearn import mixture
X_train = rsgcat[netmask]['teff_chisq'].values.reshape(-1, 1)
clf = mixture.GaussianMixture(n_components=3, covariance_type="full")
clf.fit(X_train)
plt.scatter(rsgcat[netmask]['teff_chisq'].values, clf.score_samples(rsgcat[netmask]['teff_chisq'].values.reshape(-1, 1)));

In [ ]:
plt.scatter(rsgcat[netmask]['teff_chisq'], np.log10(rsgcat[netmask]['lum_chisq']), s=1);

### parallel mcmc

In [ ]:
testcat = pd.read_csv('data/dolphot/ngc4449/ngc4449_sil_rsgcat.csv')[:10]
load_args = {
    'gal':'ngc4449', 'procdir':'data/dolphot/ngc4449', 'photfile_path':None,
    'dm':28.02, 'dmerr':0.32, 'z':-0.50, 'trgb':('F090W', 25.11),
    'modeltype':'MARCS', 'comp':'sil',
    'keep_narrow':False, 'agbcut':False,
    'rsgcat':testcat
}
rsgloader = rsg_dataloader(**load_args)

In [ ]:
sedfit_mc = mcmcfit(rsgloader, ncores=1, verbose=True, modeltype='MARCS')
f = sedfit_mc.run_mcmc(rsgcat.iloc[5], plot=True)

In [ ]:
mc_params = np.meshgrid([3300, 1000, 0.5, 4, 3.1, 0.5], indexing='ij', sparse=True)
m = np.array([rsgloader.gen_mc_obj.model[f](mc_params).flatten()[0] for f in rsgloader.nrc_filts])